# DistilBERT — Full Training Dataset (Safe Separate Experiment)

Adapted from your uploaded `04_DistilBERT(2).ipynb`. This notebook uses **all rows** from `dataset/processed/train.csv`, and the existing validation/test splits. It **never writes to** `models/distilbert/final_model/`, `models/distilbert/checkpoints/`, `results/metrics/distilbert_metrics.json`, or `results/xai_200/`.

**Before starting:** save this notebook in `PhishingDetection/notebooks/`, activate your existing `.venv` kernel, connect power, and close other GPU-heavy applications. Run the benchmark first. The final test set should be evaluated only once after model selection.

Training from scratch here means *fine-tuning pretrained DistilBERT*, not random initialization. The new model starts from `distilbert-base-uncased`, exactly like your original notebook.

In [12]:
import os, random, json, time, math, inspect
from pathlib import Path
import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "dataset" / "processed"
EXPERIMENT_DIR = PROJECT_ROOT / "models" / "distilbert" / "full_dataset_experiment"
CHECKPOINT_DIR = EXPERIMENT_DIR / "checkpoints"
FINAL_MODEL_DIR = EXPERIMENT_DIR / "final_model"
RESULTS_DIR = PROJECT_ROOT / "results" / "distilbert_full_dataset"
for path in [EXPERIMENT_DIR, CHECKPOINT_DIR, RESULTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 96
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
EPOCHS = 1  # start with one; increase only if benchmark + validation justify it
LEARNING_RATE = 2e-5
EVAL_BATCH_SIZE = 4
USE_FULL_VALIDATION = True
# Do not change to True until you have selected the best model and are ready for the final test.
RUN_FINAL_TEST = True
# Set to a checkpoint folder path ONLY when intentionally resuming an interrupted run.
RESUME_CHECKPOINT = None

print("Project root:", PROJECT_ROOT)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Not available")
print("CUDA:", torch.cuda.is_available(), "VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3,2) if torch.cuda.is_available() else "N/A")
assert (DATA_DIR / "train.csv").exists(), f"Missing {(DATA_DIR / 'train.csv')}"
assert (DATA_DIR / "val.csv").exists(), f"Missing {(DATA_DIR / 'val.csv')}"
assert (DATA_DIR / "test.csv").exists(), f"Missing {(DATA_DIR / 'test.csv')}"
assert torch.cuda.is_available(), "Full training on CPU would be extremely slow; enable CUDA before continuing."
print("Experiment output:", EXPERIMENT_DIR)


Project root: C:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection
GPU: NVIDIA GeForce GTX 1650 with Max-Q Design
CUDA: True VRAM GB: 4.0
Experiment output: C:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection\models\distilbert\full_dataset_experiment


## 1. Load original splits (no resplitting, no sampling)

In [2]:
train_df = pd.read_csv(DATA_DIR / "train.csv", usecols=["clean_url", "label_binary"])
val_df = pd.read_csv(DATA_DIR / "val.csv", usecols=["clean_url", "label_binary"])

def prepare(df, name):
    assert df[["clean_url", "label_binary"]].notna().all().all(), f"{name} contains missing values"
    assert df["label_binary"].isin([0, 1]).all(), f"{name} contains unexpected labels"
    result = df.rename(columns={"clean_url":"text", "label_binary":"labels"}).copy()
    result["text"] = result["text"].astype(str)
    result["labels"] = result["labels"].astype("int64")
    return result.reset_index(drop=True)

train_sample = prepare(train_df, "train")
val_sample = prepare(val_df, "validation")
print("Train:", len(train_sample), "Validation:", len(val_sample))
print("Train labels:\n", train_sample["labels"].value_counts().sort_index())
print("Validation labels:\n", val_sample["labels"].value_counts().sort_index())
assert len(train_sample) > 10000, "Expected full training dataset, not the old 10k sample"
assert set(train_sample["text"]).isdisjoint(set(val_sample["text"])), "Train/validation URL overlap found! Investigate before training."


Train: 448779 Validation: 96167
Train labels:
 labels
0    299652
1    149127
Name: count, dtype: int64
Validation labels:
 labels
0    64211
1    31956
Name: count, dtype: int64


## 2. Tokenizer and datasets

In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_urls(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

train_dataset = Dataset.from_pandas(train_sample, preserve_index=False).map(
    tokenize_urls, batched=True, remove_columns=["text"], desc="Tokenizing ALL training URLs"
)
val_dataset = Dataset.from_pandas(val_sample, preserve_index=False).map(
    tokenize_urls, batched=True, remove_columns=["text"], desc="Tokenizing validation URLs"
)
print("Tokenized train:", len(train_dataset), "Validation:", len(val_dataset))
print("Columns:", train_dataset.column_names)


Tokenizing ALL training URLs:   0%|          | 0/448779 [00:00<?, ? examples/s]

Tokenizing validation URLs:   0%|          | 0/96167 [00:00<?, ? examples/s]

Tokenized train: 448779 Validation: 96167
Columns: ['labels', 'input_ids', 'token_type_ids', 'attention_mask']


## 3. Load a new model and define evaluation metrics

In [4]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2,
    id2label={0:"Benign", 1:"Malicious"},
    label2id={"Benign":0, "Malicious":1}
)
collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple): logits = logits[0]
    preds = np.argmax(logits, axis=1)
    exp = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    probs = exp[:,1] / np.sum(exp, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    return {"accuracy":float(accuracy_score(labels,preds)),
            "precision":float(precision), "recall":float(recall), "f1":float(f1),
            "roc_auc":float(roc_auc_score(labels,probs)) if len(np.unique(labels)) == 2 else float('nan')}
print("Model initialized. Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model initialized. Parameters: 66,955,010


## 4. GPU benchmark (100 training batches)

This benchmark runs forward/backward passes on the **new model** but does not update its weights. It gives an approximate runtime estimate. Actual training may be slower due to optimizer updates, validation, checkpoint saving and thermal throttling.

If you get CUDA out-of-memory, restart the kernel, change `BATCH_SIZE = 1` in cell 1 and rerun.

In [5]:
from torch.utils.data import DataLoader
from torch.optim import AdamW

BENCHMARK_BATCHES = 100
bench_loader = DataLoader(train_dataset.select(range(min(len(train_dataset), BENCHMARK_BATCHES * BATCH_SIZE))),
                          batch_size=BATCH_SIZE, shuffle=False, collate_fn=collator)
model.to("cuda")
model.train()
if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()

# No optimizer step: benchmark forward/backward only, without changing parameters.
model.zero_grad(set_to_none=True)
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
start = time.perf_counter()
count = 0
for batch in bench_loader:
    batch = {k:v.to("cuda") for k,v in batch.items()}
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        loss = model(**batch).loss / GRADIENT_ACCUMULATION_STEPS
    loss.backward()
    count += 1
    if count % GRADIENT_ACCUMULATION_STEPS == 0:
        model.zero_grad(set_to_none=True)
torch.cuda.synchronize()
seconds = time.perf_counter() - start
model.zero_grad(set_to_none=True)
print(f"Benchmark: {count} batches in {seconds:.1f} sec ({seconds/count:.3f} sec/batch)")
print("Peak GPU allocated memory (GB):", round(torch.cuda.max_memory_allocated()/1024**3,2))
train_batches = math.ceil(len(train_dataset)/BATCH_SIZE)
estimated_hours = (seconds/count) * train_batches * EPOCHS / 3600
print(f"LOWER-BOUND estimated training time: {estimated_hours:.1f} hours + optimizer/evaluation/checkpoints")
print("Do not proceed until you are comfortable with this estimate.")


Benchmark: 100 batches in 12.3 sec (0.123 sec/batch)
Peak GPU allocated memory (GB): 0.62
LOWER-BOUND estimated training time: 7.7 hours + optimizer/evaluation/checkpoints
Do not proceed until you are comfortable with this estimate.


## 5. Trainer configuration — writes ONLY to the new experiment folder

In [6]:
# After benchmarking, reset the model to pretrained weights to ensure a clean experiment.
del model
import gc
gc.collect()
torch.cuda.empty_cache()
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2,
    id2label={0:"Benign",1:"Malicious"}, label2id={"Benign":0,"Malicious":1}
)

args_kwargs = dict(
    output_dir=str(CHECKPOINT_DIR), num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    weight_decay=0.01,
    eval_strategy="epoch", save_strategy="epoch", logging_strategy="steps",
    logging_steps=500, load_best_model_at_end=True,
    metric_for_best_model="f1", greater_is_better=True,
    fp16=True, report_to="none", save_total_limit=2,
    gradient_checkpointing=True,
    dataloader_num_workers=0, seed=SEED,
)
# Compatibility with older transformers versions that use evaluation_strategy.
if "eval_strategy" not in inspect.signature(TrainingArguments.__init__).parameters:
    args_kwargs["evaluation_strategy"] = args_kwargs.pop("eval_strategy")
training_args = TrainingArguments(**args_kwargs)
trainer_kwargs = dict(model=model, args=training_args, train_dataset=train_dataset,
                      eval_dataset=val_dataset, data_collator=collator,
                      compute_metrics=compute_metrics)
if "processing_class" in inspect.signature(Trainer.__init__).parameters:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer
trainer = Trainer(**trainer_kwargs)
print("Checkpoints:", CHECKPOINT_DIR)
print("Steps per epoch (approx):", math.ceil(len(train_dataset)/(BATCH_SIZE*GRADIENT_ACCUMULATION_STEPS)))


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Checkpoints: C:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection\models\distilbert\full_dataset_experiment\checkpoints
Steps per epoch (approx): 28049


## 6. Start or resume full training

**Warning:** This cell may run for many hours. To resume after an interruption, set `RESUME_CHECKPOINT` in the first cell to the exact existing `checkpoint-NNNN` folder, restart the kernel, and rerun all cells. Do not restart training from scratch unless intended.

In [7]:
print("START TRAINING", len(train_dataset), "training URLs;", EPOCHS, "epoch(s)")
start = time.time()
train_result = trainer.train(resume_from_checkpoint=str(RESUME_CHECKPOINT) if RESUME_CHECKPOINT else None)
training_time = time.time()-start
print("Training finished. Session seconds:", round(training_time,2))
print("Training loss:", train_result.training_loss)
print("Best validation F1:", trainer.state.best_metric)


START TRAINING 448779 training URLs; 1 epoch(s)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc
1,0.384102,0.059319,0.989154,0.987110,0.980160,0.983623,0.999161


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training finished. Session seconds: 31414.66
Training loss: 0.6448966310582631
Best validation F1: 0.9836230313878813


## 7. Save the new model, tokenizer, training metadata and validation metrics

In [8]:
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(FINAL_MODEL_DIR))
tokenizer.save_pretrained(str(FINAL_MODEL_DIR))
validation_metrics = trainer.evaluate()
summary = {
    "model": MODEL_NAME,
    "training_samples": len(train_dataset), "validation_samples":len(val_dataset),
    "epochs_configured": EPOCHS, "max_length":MAX_LENGTH,
    "batch_size":BATCH_SIZE,"gradient_accumulation_steps":GRADIENT_ACCUMULATION_STEPS,
    "session_training_time_seconds":round(training_time,2),
    "best_validation_f1":trainer.state.best_metric,
    "validation_metrics": validation_metrics,
    "model_directory":str(FINAL_MODEL_DIR),
}
with open(RESULTS_DIR / "training_summary.json","w",encoding="utf-8") as f:
    json.dump(summary,f,indent=2,default=float)
print("Saved NEW model:", FINAL_MODEL_DIR)
print("Saved NEW results:", RESULTS_DIR / "training_summary.json")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1,Roc Auc
0.384102,0.059319,1,0.989154,0.987110,0.980160,0.983623,0.999161


Saved NEW model: C:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection\models\distilbert\full_dataset_experiment\final_model
Saved NEW results: C:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection\results\distilbert_full_dataset\training_summary.json


## 8. Optional final test — only after model selection

Keep `RUN_FINAL_TEST = False` until your final model has been chosen using validation data. This is the original held-out test set, not a new split. The resulting metrics are saved separately and do not overwrite your previous DistilBERT metrics.

In [13]:
if not RUN_FINAL_TEST:
    print("Final test SKIPPED. To run once after model selection, set RUN_FINAL_TEST=True in the first cell, then run this cell.")
else:
    from sklearn.metrics import confusion_matrix
    test_df = pd.read_csv(DATA_DIR / "test.csv", usecols=["clean_url","label_binary"])
    test_sample = prepare(test_df, "test")
    assert set(train_sample["text"]).isdisjoint(set(test_sample["text"])), "Train/test overlap!"
    assert set(val_sample["text"]).isdisjoint(set(test_sample["text"])), "Val/test overlap!"
    test_dataset = Dataset.from_pandas(test_sample,preserve_index=False).map(
        tokenize_urls, batched=True, remove_columns=["text"], desc="Tokenizing held-out test")
    test_start=time.time()
    out=trainer.predict(test_dataset)
    elapsed=time.time()-test_start
    logits=out.predictions[0] if isinstance(out.predictions,tuple) else out.predictions
    labels=out.label_ids
    pred=np.argmax(logits,axis=1)
    metrics=compute_metrics((logits,labels))
    metrics.update({"test_samples":len(labels),"inference_time_seconds":round(elapsed,2),
                    "training_samples":len(train_dataset),"confusion_matrix":confusion_matrix(labels,pred).tolist()})
    with open(RESULTS_DIR / "test_metrics.json","w",encoding="utf-8") as f:
        json.dump(metrics,f,indent=2)
    print(json.dumps(metrics,indent=2))
    print("Saved to", RESULTS_DIR / "test_metrics.json")


Tokenizing held-out test:   0%|          | 0/96167 [00:00<?, ? examples/s]

{
  "accuracy": 0.9897885969199414,
  "precision": 0.9867982649148174,
  "recall": 0.98241331831268,
  "f1": 0.9846009095185824,
  "roc_auc": 0.9992265223277108,
  "test_samples": 96167,
  "inference_time_seconds": 1681.57,
  "training_samples": 448779,
  "confusion_matrix": [
    [
      63791,
      420
    ],
    [
      562,
      31394
    ]
  ]
}
Saved to C:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection\results\distilbert_full_dataset\test_metrics.json


## 9. How to compare with your original FYP

- **Original model:** `models/distilbert/final_model/` (10,000 training URLs)
- **New model:** `models/distilbert/full_dataset_experiment/final_model/` (all training URLs)
- **Original XAI:** `results/xai_200/` — applies to the original model only
- **New metrics:** `results/distilbert_full_dataset/` — independent experiment

If you choose the new model for the final MVP, you must rerun IG/LIME/SHAP experiments for that model and update the app model path. If you keep the original model in the MVP, you can report full-data fine-tuning as an additional experiment, clearly labeling which model each result belongs to.

**Hardware caution:** Keep the laptop plugged in and ventilated; check GPU temperature. The benchmark is only a rough estimate, and full validation can add substantial time.

In [16]:
from pathlib import Path
from transformers import AutoConfig

# Move from notebooks/ to the project root
PROJECT_ROOT = Path.cwd().parent

MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "distilbert"
    / "full_dataset_experiment"
    / "final_model"
)

print("Model path:", MODEL_PATH)
print("Model exists:", MODEL_PATH.is_dir())

config = AutoConfig.from_pretrained(
    str(MODEL_PATH),
    local_files_only=True
)

print("id2label:", config.id2label)
print("label2id:", config.label2id)

Model path: c:\Users\MSii\Desktop\NIBM\3rd Year Matrerial\Project\PhishingDetection\models\distilbert\full_dataset_experiment\final_model
Model exists: True
id2label: {0: 'Benign', 1: 'Malicious'}
label2id: {'Benign': 0, 'Malicious': 1}


In [17]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

tokenizer = AutoTokenizer.from_pretrained(
    str(MODEL_PATH),
    local_files_only=True
)

model = AutoModelForSequenceClassification.from_pretrained(
    str(MODEL_PATH),
    local_files_only=True
).to(device)

model.eval()

test_urls = [
    "https://www.google.com",
    "https://google.com",
    "http://www.google.com",
    "google.com",
    "https://example.com",
    "https://www.wikipedia.org"
]

for url in test_urls:
    inputs = tokenizer(
        url,
        return_tensors="pt",
        truncation=True,
        max_length=96
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    with torch.inference_mode():
        logits = model(**inputs).logits
        probabilities = torch.softmax(logits, dim=-1)[0]

    predicted_class = probabilities.argmax().item()

    print(f"\nURL: {url}")
    print("Prediction:", model.config.id2label[predicted_class])
    print(f"Benign: {probabilities[0].item():.4%}")
    print(f"Malicious: {probabilities[1].item():.4%}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


URL: https://www.google.com
Prediction: Malicious
Benign: 0.0003%
Malicious: 99.9997%

URL: https://google.com
Prediction: Malicious
Benign: 0.0004%
Malicious: 99.9996%

URL: http://www.google.com
Prediction: Malicious
Benign: 0.0003%
Malicious: 99.9997%

URL: google.com
Prediction: Malicious
Benign: 0.0134%
Malicious: 99.9866%

URL: https://example.com
Prediction: Malicious
Benign: 0.0004%
Malicious: 99.9996%

URL: https://www.wikipedia.org
Prediction: Malicious
Benign: 0.0003%
Malicious: 99.9997%


In [18]:
import pandas as pd
import torch
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

test_path = PROJECT_ROOT / "dataset" / "processed" / "test.csv"

df = pd.read_csv(test_path)

print("Columns:", df.columns.tolist())
print("Dataset shape:", df.shape)
print(df.head(3))

Columns: ['url', 'type', 'label_binary', 'label_multi', 'clean_url']
Dataset shape: (96167, 5)
                               url    type  label_binary  label_multi  \
0        imdb.com/title/tt0178776/  benign             0            0   
1   sports.yahoo.com/nfl/teams/ten  benign             0            0   
2  youtube.com/watch?v=JU7mribhGW8  benign             0            0   

                         clean_url  
0        imdb.com/title/tt0178776/  
1   sports.yahoo.com/nfl/teams/ten  
2  youtube.com/watch?v=JU7mribhGW8  


In [19]:
benign = df[df["label_binary"] == 0].sample(
    n=20, random_state=42
)

malicious = df[df["label_binary"] == 1].sample(
    n=20, random_state=42
)

samples = pd.concat([benign, malicious])

results = []

for _, row in samples.iterrows():
    url = str(row["url"])

    inputs = tokenizer(
        url,
        return_tensors="pt",
        truncation=True,
        max_length=96
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    with torch.inference_mode():
        probs = torch.softmax(
            model(**inputs).logits,
            dim=-1
        )[0]

    pred = int(probs.argmax().item())

    results.append({
        "url": url,
        "actual": int(row["label_binary"]),
        "predicted": pred,
        "malicious_probability": float(probs[1].item())
    })

result_df = pd.DataFrame(results)

print(result_df.to_string(index=False))
print("\nSample accuracy:",
      (result_df["actual"] == result_df["predicted"]).mean())

print("\nConfusion table:")
print(pd.crosstab(
    result_df["actual"],
    result_df["predicted"],
    rownames=["Actual"],
    colnames=["Predicted"],
    dropna=False
))

                                                                                                                                                url  actual  predicted  malicious_probability
                                                                                                                   platteriverdecoyco.com/demo.html       0          0               0.000176
                                                                                                                                palzoo.net/TomHanks       0          0               0.002914
 corporate.univision.com/2007/05/disney-abc-international-television-latin-america-and-univision-sign-unprecedented-strategic-production-agreement/       0          0               0.000012
                                                                                                          lib.o56.ru/lat/ASPRIN/Thieves_World-2.txt       0          0               0.000079
                                                  

In [20]:
import pandas as pd
import torch

# Check URL formats in the held-out dataset
for label, name in [(0, "Benign"), (1, "Malicious")]:
    subset = df[df["label_binary"] == label]
    urls = subset["url"].fillna("").astype(str)

    has_scheme = urls.str.match(
        r"(?i)^https?://"
    )

    print(f"\n{name}")
    print("Total URLs:", len(urls))
    print("With HTTP/HTTPS:", has_scheme.sum())
    print("Without HTTP/HTTPS:", (~has_scheme).sum())
    print("Scheme percentage:", round(has_scheme.mean() * 100, 2))
    print("Average URL length:", round(urls.str.len().mean(), 2))

# Compare model predictions on different formats
test_urls = [
    "google.com",
    "www.google.com",
    "https://www.google.com",
    "google.com/search?q=machine+learning",
    "www.google.com/search?q=machine+learning",
    "https://www.google.com/search?q=machine+learning",
    "wikipedia.org",
    "en.wikipedia.org/wiki/Artificial_intelligence",
    "https://en.wikipedia.org/wiki/Artificial_intelligence",
    "youtube.com/watch?v=JU7mribhGW8",
    "https://youtube.com/watch?v=JU7mribhGW8",
]

rows = []

for url in test_urls:
    inputs = tokenizer(
        url,
        return_tensors="pt",
        truncation=True,
        max_length=96
    ).to(device)

    with torch.inference_mode():
        probs = model(**inputs).logits.softmax(dim=-1)[0]

    rows.append({
        "URL": url,
        "Prediction": "Malicious" if probs.argmax().item() == 1 else "Benign",
        "P(Malicious)": round(probs[1].item(), 6)
    })

display(pd.DataFrame(rows))


Benign
Total URLs: 64211
With HTTP/HTTPS: 5241
Without HTTP/HTTPS: 58970
Scheme percentage: 8.16
Average URL length: 57.32

Malicious
Total URLs: 31956
With HTTP/HTTPS: 21434
Without HTTP/HTTPS: 10522
Scheme percentage: 67.07
Average URL length: 64.19


,URL,Prediction,P(Malicious)
0,google.com,Malicious,0.999866
1,www.google.com,Malicious,0.999087
2,https://www.google.com,Malicious,0.999997
3,google.com/search?q=machine+learning,Benign,0.000526
4,www.google.com/search?q=machine+learning,Malicious,0.999974
5,https://www.google.com/search?q=machine+learning,Malicious,0.999994
6,wikipedia.org,Malicious,0.999892
7,en.wikipedia.org/wiki/Artificial_intelligence,Malicious,0.994961
8,https://en.wikipedia.org/wiki/Artificial_intel...,Malicious,0.999983
9,youtube.com/watch?v=JU7mribhGW8,Benign,0.000003


In [21]:
import pandas as pd
import torch

# Select held-out URLs without HTTP/HTTPS
no_scheme = df[
    ~df["url"].astype(str).str.match(r"(?i)^https?://")
].copy()

benign = no_scheme[no_scheme["label_binary"] == 0].sample(
    n=100, random_state=42
)

malicious = no_scheme[no_scheme["label_binary"] == 1].sample(
    n=100, random_state=42
)

samples = pd.concat([benign, malicious], ignore_index=True)

def get_malicious_probability(url):
    inputs = tokenizer(
        url,
        return_tensors="pt",
        truncation=True,
        max_length=96
    ).to(device)

    with torch.inference_mode():
        probs = model(**inputs).logits.softmax(dim=-1)[0]

    return float(probs[1].item())

results = []

for _, row in samples.iterrows():
    original = str(row["url"])
    modified = "https://" + original

    p_original = get_malicious_probability(original)
    p_modified = get_malicious_probability(modified)

    results.append({
        "url": original,
        "actual_label": int(row["label_binary"]),
        "original_probability": p_original,
        "https_probability": p_modified,
        "original_prediction": int(p_original >= 0.5),
        "https_prediction": int(p_modified >= 0.5),
        "prediction_flipped": (p_original >= 0.5) != (p_modified >= 0.5),
        "probability_change": p_modified - p_original
    })

results_df = pd.DataFrame(results)

print("Total URLs:", len(results_df))
print("Prediction flip rate:",
      round(results_df["prediction_flipped"].mean() * 100, 2), "%")

print("Average absolute probability change:",
      round(results_df["probability_change"].abs().mean(), 4))

print("\nResults by actual class:")
print(results_df.groupby("actual_label").agg(
    flip_rate=("prediction_flipped", "mean"),
    mean_probability_change=("probability_change", "mean"),
    mean_absolute_change=("probability_change", lambda x: x.abs().mean())
))

results_df.to_csv(
    PROJECT_ROOT / "results" / "https_prefix_robustness.csv",
    index=False
)

Total URLs: 200
Prediction flip rate: 37.0 %
Average absolute probability change: 0.3709

Results by actual class:
              flip_rate  mean_probability_change  mean_absolute_change
actual_label                                                          
0                  0.70                 0.700998              0.700999
1                  0.04                 0.040727              0.040727
